# Multi-Layer Perceptron (MLP)

## 1. Introduction

Neural networks are among the most important machine learning models used today. They are inspired by the functioning of the human brain and are designed to learn complex relationships between data.

The **Multi-Layer Perceptron (MLP)** is widely used for:

* ✅ regression tasks,
* ✅ classification tasks,
* ✅ prediction systems,
* ✅ pattern recognition.

In this notebook, we study the implementation of an MLP from `ifri_mini_ml_lib` and compare it with scikit-learn's implementation.

## 2. Key Concepts

A Multi-Layer Perceptron is a **feedforward neural network** composed of:

* an **input layer** with $n$ features,
* one or more **hidden layers** with learnable weights and biases,
* an **output layer** with **softmax activation** (for classification) or linear activation (for regression).

### Classification

For classification tasks:
- The output layer uses **softmax** to produce a probability distribution over classes
- The loss function is **categorical cross-entropy**
- Prediction is the class with the highest probability

$$P(y=k|x) = \text{softmax}(W_L \cdot \text{ReLU}(W_{L-1} \cdots \text{ReLU}(W_1 x + b_1) + b_{L-1}) + b_L)$$

### Regression

For regression tasks:
- The output layer uses **linear activation** (no activation)
- The loss function is **mean squared error (MSE)**
- Prediction is the continuous output value

$$\hat{y} = W_L \cdot \text{ReLU}(W_{L-1} \cdots \text{ReLU}(W_1 x + b_1) + b_{L-1}) + b_L$$

### Activation Functions

Common activation functions include:
- **ReLU (Rectified Linear Unit)**: $f(x) = \max(0, x)$ - computationally efficient
- **Sigmoid**: $f(x) = \frac{1}{1 + e^{-x}}$ - outputs between 0 and 1
- **Tanh**: $f(x) = \tanh(x)$ - outputs between -1 and 1
- **Leaky ReLU**: $f(x) = \begin{cases} x & \text{if } x > 0 \\ 0.01x & \text{otherwise} \end{cases}$

## 3. Training Algorithm

The MLP is trained using **backpropagation** with various optimization algorithms:

1. **Forward Pass**: Compute activations layer by layer
2. **Loss Computation**: Calculate loss (cross-entropy for classification, MSE for regression)
3. **Backward Pass**: Compute gradients using chain rule
4. **Weight Update**: Update weights using optimizers (SGD, Adam, RMSprop, Momentum)

Optimization algorithms available:
- **SGD (Stochastic Gradient Descent)**: $w \leftarrow w - \eta \nabla L$
- **Momentum**: Accelerates convergence with velocity accumulation
- **Adam**: Adaptive learning rates with momentum and RMSprop
- **RMSprop**: Adapts learning rate based on gradient history

Early stopping can be used to prevent overfitting by monitoring validation loss.

## 4. Implementation

For this implementation, we will use two datasets:

1. **Iris** for classification
2. **Diabetes** for regression

We use the preprocessing modules from `ifri_mini_ml_lib` (StandardScaler, DataSplitter) instead of scikit-learn.


In [17]:
from time import perf_counter

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris, load_diabetes
from sklearn.neural_network import MLPClassifier as SKMLPClassifier
from sklearn.neural_network import MLPRegressor as SKMLPRegressor
from ifri_mini_ml_lib.neural_networks import MLPClassifier, MLPRegressor
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter
from ifri_mini_ml_lib.preprocessing.preparation.scaler.standard_scaler import StandardScaler
from ifri_mini_ml_lib.metrics.classification import accuracy, f1_score, recall, precision
from ifri_mini_ml_lib.metrics.regression import evaluate_rg_model





### Classification with Iris Dataset


In [18]:
# Load iris dataset
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")
X.head()


Dataset shape: (150, 4)
Number of classes: 3


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [19]:
# Split and scale data using ifri_mini_ml_lib
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (120, 4)
Test set shape: (30, 4)


In [20]:
# Train ifri_mini_ml_lib MLPClassifier
start_custom = perf_counter()
mlp_custom = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=16,
    learning_rate=0.001,
    max_iter=150,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
)
mlp_custom.fit(X_train, y_train)
time_custom = perf_counter() - start_custom

y_pred_custom = mlp_custom.predict(X_test)

In [21]:
# Train scikit-learn MLPClassifier
start_sklearn = perf_counter()
mlp_sklearn = SKMLPClassifier(
    hidden_layer_sizes=(50,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=16,
    learning_rate_init=0.001,
    max_iter=150,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
)
mlp_sklearn.fit(X_train, y_train)
time_sklearn = perf_counter() - start_sklearn

y_pred_sklearn = mlp_sklearn.predict(X_test)



In [22]:
# Compare classification results
classification_results = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score', 'Recall', 'Precision', 'Time'],
    'ifri_mini_ml_lib': [
        accuracy(y_test, y_pred_custom),
        f1_score(y_test, y_pred_custom),
        recall(y_test, y_pred_custom),
        precision(y_test, y_pred_custom),
        time_custom
    ],
    'scikit-learn': [
        accuracy(y_test, y_pred_sklearn),
        f1_score(y_test, y_pred_sklearn),
        recall(y_test, y_pred_sklearn),
        precision(y_test, y_pred_sklearn),
        time_sklearn
    ],
})

print(classification_results.to_markdown(index=False))


| Metric    |   ifri_mini_ml_lib |   scikit-learn |
|:----------|-------------------:|---------------:|
| Accuracy  |           1        |       0.866667 |
| F1 Score  |           1        |       0.714286 |
| Recall    |           1        |       0.555556 |
| Precision |           1        |       1        |
| Time      |           0.495832 |       0.182305 |


### Regression with Diabetes Dataset


In [23]:
# Load diabetes dataset
diabetes = load_diabetes(as_frame=True)
X_reg = diabetes.data
y_reg = diabetes.target

print(f"Dataset shape: {X_reg.shape}")
print(f"Target range: [{y_reg.min():.2f}, {y_reg.max():.2f}]")
X_reg.head()


Dataset shape: (442, 10)
Target range: [25.00, 346.00]


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [24]:
# Split and scale regression data (single shared splitter already defined)
splitter = DataSplitter(seed=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = splitter.train_test_split(
    X_reg, y_reg, test_size=0.2
)

scaler_reg = StandardScaler()
X_train_reg = scaler_reg.fit_transform(X_train_reg)
X_test_reg = scaler_reg.transform(X_test_reg)

print(f"Training set shape: {X_train_reg.shape}")
print(f"Test set shape: {X_test_reg.shape}")


Training set shape: (354, 10)
Test set shape: (88, 10)


In [25]:
# Train ifri_mini_ml_lib MLPRegressor and evaluate using evaluate_rg_model
start_custom_reg = perf_counter()
mlp_regressor_custom = MLPRegressor(
    hidden_layer_sizes=(50,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=16,
    learning_rate=0.001,
    max_iter=150,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
)
mlp_regressor_custom.fit(X_train_reg, y_train_reg)
time_custom_reg = perf_counter() - start_custom_reg

y_pred_custom_reg = mlp_regressor_custom.predict(X_test_reg)
metrics_custom = evaluate_rg_model(y_test_reg.to_numpy(), y_pred_custom_reg)


In [26]:
# Train scikit-learn MLPRegressor and evaluate using evaluate_rg_model
start_sklearn_reg = perf_counter()
mlp_regressor_sklearn = SKMLPRegressor(
    hidden_layer_sizes=(50,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=16,
    learning_rate_init=0.001,
    max_iter=150,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
)
mlp_regressor_sklearn.fit(X_train_reg, y_train_reg)
time_sklearn_reg = perf_counter() - start_sklearn_reg

y_pred_sklearn_reg = mlp_regressor_sklearn.predict(X_test_reg)
metrics_sklearn = evaluate_rg_model(y_test_reg.to_numpy() , y_pred_sklearn_reg)

C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


In [27]:
# Compare regression results using evaluate_rg_model outputs
regression_results = pd.DataFrame({
    'Metric': ['MAE', 'MAPE', 'MSE', 'RMSE', 'R²', 'Time'],
    'ifri_mini_ml_lib': [
        metrics_custom['MAE'],
        metrics_custom['MAPE'],
        metrics_custom['MSE'],
        metrics_custom['RMSE'],
        metrics_custom['R²'],
        time_custom_reg
    ],
    'scikit-learn': [
        metrics_sklearn['MAE'],
        metrics_sklearn['MAPE'],
        metrics_sklearn['MSE'],
        metrics_sklearn['RMSE'],
        metrics_sklearn['R²'],
        time_sklearn_reg
    ],
})

print(regression_results.to_markdown(index=False))


| Metric   |   ifri_mini_ml_lib |   scikit-learn |
|:---------|-------------------:|---------------:|
| MAE      |          44.1168   |      45.447    |
| MAPE     |          37.7358   |      39.015    |
| MSE      |        3100.91     |    3292.13     |
| RMSE     |          55.6858   |      57.3771   |
| R²       |           0.419475 |       0.383677 |
| Time     |           1.22626  |       1.72164  |


## 6. Advantages and Limitations

### Advantages
- Flexible architecture: can approximate any continuous function with sufficient hidden units
- Effective for non-linear relationships in data
- Multiple optimization algorithms available (SGD, Adam, RMSprop)
- Early stopping prevents overfitting

### Limitations
- Computational cost increases with network size and dataset size
- Prone to overfitting on small datasets
- Requires careful hyperparameter tuning (learning rate, layer sizes, regularization)
- Interpretability is challenging (black-box nature)
- Sensitive to feature scaling and initialization

### Best Practices
1. Normalize/scale features using `StandardScaler` or `MinMaxScaler`
2. Use early stopping to prevent overfitting
3. Start with simple architectures and increase complexity gradually
4. Use cross-validation to evaluate model robustness
5. Monitor loss history during training
